# Notebook 3: Generate Reports

**VLM-ARB Cloud Framework**

This notebook fetches results from Firestore and generates comprehensive reports with visualizations.

## Workflow
1. Setup: Authenticate with Firebase
2. Fetch evaluation results from Firestore
3. Generate visualizations (bar charts, heatmaps)
4. Create PDF reports
5. Upload reports to Cloud Storage
6. Generate shareable links for team

**Key Feature**: Shareable public links - all team members can view reports without additional auth.

---

## Cell 1: Install Dependencies & Setup

In [ ]:
# Install dependencies
import subprocess
import sys

packages = [
    'firebase-admin',
    'matplotlib',
    'seaborn',
    'numpy',
    'pandas',
    'reportlab',
    'pillow',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print("✅ All dependencies installed")

## Cell 2: Setup Authentication & Fetch Results

In [ ]:
## Cell 2: Setup & Fetch Results

import sys
import json
sys.path.insert(0, '/root')

from pathlib import Path
from datetime import datetime
import logging

# Defaults used by later cells
pdf_url = None
chart_url = None
json_url = None
selected_run = None
selected_run_id = None
metrics = {}
runs = []
normalized_runs = []

# Mount Drive when running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# Bootstrap utilities path for Colab runtime
team_folder = Path('/content/drive/MyDrive/VLM-ARB-Team')
context = {'creds_path': str(team_folder / 'secrets' / 'serviceAccountKey.json')}
gpu_info = {}

try:
    colab_utilities = Path('/root/utilities')
    colab_utilities.mkdir(parents=True, exist_ok=True)
    (colab_utilities / '__init__.py').write_text('# Utilities module\n', encoding='utf-8')

    source_utilities = team_folder / 'utilities'
    if source_utilities.exists():
        import shutil
        shutil.copytree(source_utilities, colab_utilities, dirs_exist_ok=True)
except Exception as e:
    print(f"⚠️ Utilities bootstrap warning: {str(e)[:120]}")

# Quick setup (if helper exists)
try:
    from utilities.auth_helpers import quick_colab_setup
    team_folder, context, gpu_info = quick_colab_setup()
except Exception:
    pass

# Initialize Firebase
try:
    from utilities.cloud_sync import FirebaseSync
    fs = FirebaseSync(context['creds_path'])
    print('✅ Firebase initialized')
except Exception as e:
    fs = None
    print(f"⚠️ Firebase unavailable: {str(e)[:120]}")


def _to_float(value, default=0.0):
    try:
        return float(value)
    except Exception:
        return float(default)


def _normalize_model_metrics(payload: dict) -> dict:
    """Return schema: {model: {asr, ods, sbr, ...}} from multiple source formats."""
    if not isinstance(payload, dict):
        return {}

    # Notebook 2 Firestore upload shape: metrics.results[model]
    if isinstance(payload.get('results'), dict):
        results_block = payload['results']
        normalized = {}

        for model_name, mm in results_block.items():
            if not isinstance(mm, dict):
                continue

            # Preferred ASR/ODS/SBR shape
            if 'asr' in mm or 'ods' in mm or 'sbr' in mm:
                normalized[model_name] = {
                    'asr': _to_float(mm.get('asr', 0.0)),
                    'ods': _to_float(mm.get('ods', 0.0)),
                    'sbr': _to_float(mm.get('sbr', 0.0)),
                    'pairs': mm.get('pairs', mm.get('num_pairs', 0)),
                }
                continue

            # Local robustness shape: prediction_change_rate, accuracy_drop_abs, target_hit_rate
            if 'prediction_change_rate' in mm:
                normalized[model_name] = {
                    'asr': _to_float(mm.get('prediction_change_rate', 0.0)),
                    'ods': _to_float(mm.get('accuracy_drop_abs', 0.0)),
                    'sbr': _to_float(mm.get('target_hit_rate', 0.0)),
                    'pairs': mm.get('num_pairs', 0),
                }

        return normalized

    # Direct shape: metrics[model]
    direct = {}
    for k, v in payload.items():
        if not isinstance(v, dict):
            continue

        if 'asr' in v or 'ods' in v or 'sbr' in v:
            direct[k] = {
                'asr': _to_float(v.get('asr', 0.0)),
                'ods': _to_float(v.get('ods', 0.0)),
                'sbr': _to_float(v.get('sbr', 0.0)),
                'pairs': v.get('pairs', v.get('num_pairs', 0)),
            }
        elif 'prediction_change_rate' in v:
            direct[k] = {
                'asr': _to_float(v.get('prediction_change_rate', 0.0)),
                'ods': _to_float(v.get('accuracy_drop_abs', 0.0)),
                'sbr': _to_float(v.get('target_hit_rate', 0.0)),
                'pairs': v.get('num_pairs', 0),
            }

    return direct


def _extract_model_metrics(run_doc: dict) -> dict:
    if not isinstance(run_doc, dict):
        return {}

    # Firestore run doc usually has a top-level metrics object
    if isinstance(run_doc.get('metrics'), dict):
        return _normalize_model_metrics(run_doc['metrics'])

    # Saved local artifact may already be {'run_id':..., 'metrics': {...}}
    if isinstance(run_doc.get('metrics'), dict):
        return _normalize_model_metrics(run_doc['metrics'])

    # Last resort: treat entire dict as payload
    return _normalize_model_metrics(run_doc)


print('\n📊 Fetching evaluation results from Firestore...')
if fs and not fs.offline_mode:
    try:
        runs = fs.list_runs(collection='results', limit=100)
        print(f"✅ Found {len(runs)} run documents")
    except Exception as e:
        print(f"⚠️ Firestore fetch failed: {str(e)[:120]}")
else:
    print('⚠️ Firestore unavailable, trying local fallback files...')

# Build normalized run list from Firestore docs
for run in runs:
    model_metrics = _extract_model_metrics(run)
    if not model_metrics:
        continue

    normalized_runs.append({
        'run_id': run.get('run_id', f"run_{len(normalized_runs)+1}"),
        'timestamp': str(run.get('timestamp', '')),
        'dataset_version': run.get('metrics', {}).get('dataset_version', run.get('dataset_version', 'unknown')),
        'model_metrics': model_metrics,
        'raw': run,
    })

# Local JSON fallback if Firestore returned nothing usable
if not normalized_runs:
    candidate_dirs = [
        team_folder / 'reports',
        team_folder / 'results',
        Path('/content/drive/MyDrive/VLM-ARB-Team/reports'),
        Path('/content/drive/MyDrive/VLM-ARB-Team/results'),
    ]

    json_files = []
    for d in candidate_dirs:
        if d.exists():
            json_files.extend(sorted(d.glob('*.json'), key=lambda p: p.stat().st_mtime, reverse=True))

    seen = set()
    unique_files = []
    for p in json_files:
        if str(p) not in seen:
            seen.add(str(p))
            unique_files.append(p)

    for jf in unique_files:
        try:
            doc = json.loads(jf.read_text(encoding='utf-8'))
        except Exception:
            continue

        if not isinstance(doc, dict):
            continue

        model_metrics = _extract_model_metrics(doc)
        if not model_metrics:
            continue

        normalized_runs.append({
            'run_id': doc.get('run_id', jf.stem),
            'timestamp': datetime.fromtimestamp(jf.stat().st_mtime).isoformat(),
            'dataset_version': doc.get('dataset_version', doc.get('metrics', {}).get('dataset_version', 'local-file')),
            'model_metrics': model_metrics,
            'raw': doc,
        })

    if normalized_runs:
        print(f"✅ Loaded {len(normalized_runs)} run(s) from local JSON fallback")

# Demo fallback only if nothing was found at all
if not normalized_runs:
    demo_run_id = f"eval_demo_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    normalized_runs = [{
        'run_id': demo_run_id,
        'timestamp': datetime.now().isoformat(),
        'dataset_version': 'demo',
        'model_metrics': {
            'clip': {'asr': 0.22, 'ods': 0.14, 'sbr': 0.00, 'pairs': 50},
            'mobilevit': {'asr': 0.31, 'ods': 0.19, 'sbr': 0.00, 'pairs': 50},
            'blip2': {'asr': 0.41, 'ods': 0.29, 'sbr': 0.01, 'pairs': 50},
        },
        'raw': {},
    }]
    print('⚠️ No valid Firestore or local runs found. Using demo run for plotting.')

print(f"📋 Ready runs for plotting: {len(normalized_runs)}")

## Cell 3: Select Run(s) & Display Results

In [ ]:
import pandas as pd

# Select latest normalized run
if normalized_runs:
    selected = normalized_runs[0]
    selected_run = selected['raw']
    selected_run_id = selected['run_id']
    metrics = selected['model_metrics']

    print(f"\n📊 Selected Run: {selected_run_id}")
    print(f"   Timestamp: {selected.get('timestamp', 'unknown')}")
    print(f"   Dataset: {selected.get('dataset_version', 'unknown')}")

    rows = []
    for model_name, model_metrics in metrics.items():
        if isinstance(model_metrics, dict):
            rows.append({
                'Model': model_name,
                'ASR': float(model_metrics.get('asr', 0.0)),
                'ODS': float(model_metrics.get('ods', 0.0)),
                'SBR': float(model_metrics.get('sbr', 0.0)),
                'Pairs': int(model_metrics.get('pairs', 0)) if str(model_metrics.get('pairs', '')).isdigit() else model_metrics.get('pairs', 'n/a'),
            })

    results_df = pd.DataFrame(rows).sort_values('Model').reset_index(drop=True) if rows else pd.DataFrame()

    if not results_df.empty:
        print('\n📈 Results Summary:')
        print(results_df.to_string(index=False))
    else:
        print('⚠️ Selected run has no model metrics')

    # Build long-form table across runs for trend plots
    trend_rows = []
    for run in normalized_runs:
        for model_name, mm in run['model_metrics'].items():
            if not isinstance(mm, dict):
                continue
            trend_rows.append({
                'run_id': run['run_id'],
                'timestamp': run.get('timestamp', ''),
                'model': model_name,
                'ASR': float(mm.get('asr', 0.0)),
                'ODS': float(mm.get('ods', 0.0)),
                'SBR': float(mm.get('sbr', 0.0)),
            })

    trend_df = pd.DataFrame(trend_rows)
else:
    print('No runs available')
    selected_run = None
    selected_run_id = None
    metrics = {}
    results_df = pd.DataFrame()
    trend_df = pd.DataFrame()

## Cell 4: Generate Visualizations (Bar Charts)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

sns.set_theme(style='whitegrid')

if selected_run_id and not results_df.empty:
    # Setup output directory
    figures_dir = Path('/tmp/vlm_arb_figures')
    figures_dir.mkdir(parents=True, exist_ok=True)

    saved_figures = {}

    # Graph 1: Grouped metric bars for selected run
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(results_df))
    width = 0.25

    ax.bar(x - width, results_df['ASR'], width=width, label='ASR', color='#ff6b6b')
    ax.bar(x,         results_df['ODS'], width=width, label='ODS', color='#4ecdc4')
    ax.bar(x + width, results_df['SBR'], width=width, label='SBR', color='#ffe66d')

    ax.set_title(f'Metrics by Model | {selected_run_id}')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1)
    ax.set_xticks(x)
    ax.set_xticklabels(results_df['Model'], rotation=25, ha='right')
    ax.legend()

    plt.tight_layout()
    p1 = figures_dir / 'selected_run_grouped_metrics.png'
    plt.savefig(p1, dpi=150, bbox_inches='tight')
    plt.close()
    saved_figures['grouped_metrics'] = str(p1)

    # Graph 2: Heatmap for selected run
    heat_df = results_df.set_index('Model')[['ASR', 'ODS', 'SBR']]
    fig, ax = plt.subplots(figsize=(8, max(3, 0.8 * len(heat_df))))
    sns.heatmap(heat_df, annot=True, fmt='.3f', cmap='YlOrRd', vmin=0, vmax=1, linewidths=0.5, ax=ax)
    ax.set_title(f'Metric Heatmap | {selected_run_id}')

    plt.tight_layout()
    p2 = figures_dir / 'selected_run_heatmap.png'
    plt.savefig(p2, dpi=150, bbox_inches='tight')
    plt.close()
    saved_figures['heatmap'] = str(p2)

    # Graph 3: ASR trend across runs (if multiple runs exist)
    if not trend_df.empty and trend_df['run_id'].nunique() > 1:
        trend_df = trend_df.copy()
        trend_df['run_idx'] = trend_df.groupby('run_id').ngroup()

        fig, ax = plt.subplots(figsize=(10, 5))
        for model_name, part in trend_df.groupby('model'):
            part = part.sort_values('run_idx')
            ax.plot(part['run_idx'], part['ASR'], marker='o', label=model_name)

        run_labels = (
            trend_df[['run_idx', 'run_id']]
            .drop_duplicates()
            .sort_values('run_idx')
        )
        ax.set_xticks(run_labels['run_idx'])
        ax.set_xticklabels(run_labels['run_id'], rotation=30, ha='right')
        ax.set_ylim(0, 1)
        ax.set_ylabel('ASR')
        ax.set_title('ASR Trend Across Runs')
        ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1.0))

        plt.tight_layout()
        p3 = figures_dir / 'asr_trend_across_runs.png'
        plt.savefig(p3, dpi=150, bbox_inches='tight')
        plt.close()
        saved_figures['asr_trend'] = str(p3)

    # Graph 4: Average metrics summary bar
    avg_asr = float(results_df['ASR'].mean())
    avg_ods = float(results_df['ODS'].mean())
    avg_sbr = float(results_df['SBR'].mean())

    fig, ax = plt.subplots(figsize=(7, 4))
    names = ['ASR', 'ODS', 'SBR']
    vals = [avg_asr, avg_ods, avg_sbr]
    colors = ['#ff8787', '#63e6be', '#ffd43b']
    ax.bar(names, vals, color=colors)
    ax.set_ylim(0, 1)
    ax.set_title(f'Average Metrics | {selected_run_id}')
    ax.set_ylabel('Score')

    for i, v in enumerate(vals):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center')

    plt.tight_layout()
    p4 = figures_dir / 'average_metrics_summary.png'
    plt.savefig(p4, dpi=150, bbox_inches='tight')
    plt.close()
    saved_figures['average_summary'] = str(p4)

    print('✅ Graph generation complete')
    for k, v in saved_figures.items():
        print(f'   {k}: {v}')
else:
    print('⏭️ Skipping visualization (no selected run data)')
    figures_dir = Path('/tmp/vlm_arb_figures')
    saved_figures = {}

## Cell 5: Create PDF Report

In [ ]:
from pathlib import Path
import json
import shutil

# Create reports directory on Google Drive
drive_reports_dir = Path('/content/drive/MyDrive/VLM-ARB-Team/reports')
drive_reports_dir.mkdir(parents=True, exist_ok=True)

if selected_run_id:
    print('💾 Saving report artifacts to Google Drive...')

    # Save all generated figures
    for graph_name, graph_path in saved_figures.items():
        src = Path(graph_path)
        if src.exists():
            dst = drive_reports_dir / f"{selected_run_id}_{graph_name}.png"
            shutil.copy(str(src), str(dst))
            print(f"✅ Figure saved: {dst}")

    # Save selected run metrics JSON
    run_json_path = Path('/tmp/eval_results_selected_run.json')
    with open(run_json_path, 'w', encoding='utf-8') as f:
        json.dump({'run_id': selected_run_id, 'metrics': metrics}, f, indent=2, default=str)

    drive_json_path = drive_reports_dir / f"{selected_run_id}_results.json"
    shutil.copy(str(run_json_path), str(drive_json_path))
    print(f"✅ JSON saved: {drive_json_path}")

    # Save a compact multi-run table JSON for trend auditing
    trend_json_path = Path('/tmp/eval_results_trend_table.json')
    if 'trend_df' in globals() and not trend_df.empty:
        trend_records = trend_df.to_dict(orient='records')
    else:
        trend_records = []

    with open(trend_json_path, 'w', encoding='utf-8') as f:
        json.dump(trend_records, f, indent=2, default=str)

    drive_trend_path = drive_reports_dir / f"{selected_run_id}_trend_table.json"
    shutil.copy(str(trend_json_path), str(drive_trend_path))
    print(f"✅ Trend table saved: {drive_trend_path}")

    print('\n📂 All report artifacts saved to Google Drive:')
    print(f'   {drive_reports_dir}')
else:
    print('⏭️ No selected run to save')

## Cell 6: Upload Reports to Cloud Storage

In [ ]:
print("\n" + "="*60)
print("✅ REPORT GENERATION COMPLETE")
print("="*60)

drive_reports_dir = Path("/content/drive/MyDrive/VLM-ARB-Team/reports")

if drive_reports_dir.exists():
    print(f"\n📂 Reports saved to Google Drive:")
    print(f"   📁 {drive_reports_dir}")
    
    print(f"\n📋 Report Files:")
    for report_file in sorted(drive_reports_dir.glob(f"{selected_run_id}*")):
        print(f"   📄 {report_file.name}")
    
    print(f"\n🔗 To share reports with team:")
    print(f"   1. Open Google Drive")
    print(f"   2. Navigate to VLM-ARB-Team/reports")
    print(f"   3. Right-click on PDF file → Share")
    print(f"   4. Set access to 'Anyone with link' or 'Your team'")
    print(f"   5. Copy link and share via Slack/email")
    
    print(f"\n💡 Tip: Share the entire 'reports' folder with your team")
    print(f"   for easy access to all historical reports")
else:
    print("⚠️  Reports directory not found")

print("="*60)

## Cell 7: Generate Shareable Links & Display Results

In [ ]:
print('\n' + '='*60)
print('✅ REPORT GENERATION COMPLETE')
print('='*60)

if selected_run_id:
    print('\n📊 Report Details:')
    print(f'   Run ID: {selected_run_id}')
    print(f"   Timestamp: {datetime.now().isoformat()}")
    print(f"   Models: {len(results_df) if 'results_df' in globals() else 0}")

    print('\n📈 Generated Graphs:')
    if saved_figures:
        for graph_name, graph_path in saved_figures.items():
            print(f'   {graph_name}: {graph_path}')
    else:
        print('   No graphs were generated')

    print('\n📊 Metrics Summary:')
    for model_name, model_metrics in metrics.items():
        if isinstance(model_metrics, dict):
            print(
                f"   {model_name}: "
                f"ASR={float(model_metrics.get('asr', 0.0)):.3f}, "
                f"ODS={float(model_metrics.get('ods', 0.0)):.3f}, "
                f"SBR={float(model_metrics.get('sbr', 0.0)):.3f}"
            )

    print('\n📂 Drive Folder:')
    print('   /content/drive/MyDrive/VLM-ARB-Team/reports')
else:
    print('No results to display')

print('='*60)

## Cell 8: Optional - Browse All Historical Reports

In [ ]:
print("\n📚 All Generated Reports:")
print("\nTo view all reports, fetch from Cloud Storage:")

if fs and not fs.offline_mode:
    all_reports = fs.list_files("reports/")
    print(f"\nFound {len(all_reports)} files in Cloud Storage:")
    for report_file in all_reports[:10]:  # Show first 10
        file_name = Path(report_file).name
        url = fs.get_public_url(report_file)
        print(f"   • {file_name}")
else:
    print("Not connected to Cloud Storage")